In [4]:
# --- STEP 1: INITIAL SETUP, SPLITTING, AND FEATURE TYPE DEFINITION ---

# 🌟 1. Ensure all necessary imports are present
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectFromModel

print("=" * 70)
print("STEP 1: DATA SETUP AND SPLITTING")
print("=" * 70)

# 🌟 2. FIX: Provide the correct, absolute path to your file
# NOTE: You MUST replace the path below with the actual location of 'Loan_default.csv'
FILE_PATH = r'D:\Desktop\CreditPathAI\data\Loan_default.csv' # <--- REPLACE THIS LINE!

try:
    df = pd.read_csv(FILE_PATH) 
    print(f"✅ Data loaded successfully from: {FILE_PATH}")
except FileNotFoundError:
    print("-" * 70)
    print(f"FATAL ERROR: File '{FILE_PATH}' not found. Please verify the absolute path.")
    raise # Stop execution as the data cannot be loaded

# Define Target and Features (Adjust 'Default' if needed)
TARGET_COLUMN = 'Default'
# 'df' is now defined and can be used for dropping columns
X = df.drop([TARGET_COLUMN, 'LoanID'], axis=1, errors='ignore') 
y = df[TARGET_COLUMN]
    
# Identify Column Types
numerical_cols = X.select_dtypes(include=['int64', 'float64']).columns
categorical_cols = X.select_dtypes(include=['object', 'category']).columns

# Data Splitting
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
y_train_encoded = LabelEncoder().fit_transform(y_train)

print(f"\nData split. X_train shape: {X_train.shape}")
print(f"Numerical features identified: {len(numerical_cols)}")
print(f"Categorical features identified: {len(categorical_cols)}")
print("-" * 70)

STEP 1: DATA SETUP AND SPLITTING
✅ Data loaded successfully from: D:\Desktop\CreditPathAI\data\Loan_default.csv

Data split. X_train shape: (204277, 16)
Numerical features identified: 9
Categorical features identified: 7
----------------------------------------------------------------------


In [6]:
# --- STEP 2: FULL PREPROCESSING PIPELINE (IMPUTATION, SCALING, ENCODING) ---

# 🌟 FIX: Add the import for Pipeline
from sklearn.pipeline import Pipeline 
# (Assuming SimpleImputer, StandardScaler, OneHotEncoder, and ColumnTransformer are already imported from Step 1)

print("=" * 70)
print("STEP 2: FULL PREPROCESSING (Imputation, Scaling, Encoding)")
print("=" * 70)

# 1. Define Imputation, Scaling, and Encoding Transformers
num_pipeline = [
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
]

cat_pipeline = [
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
]

# 2. Create the ColumnTransformer
full_preprocessor = ColumnTransformer(
    transformers=[
        # 🌟 FIX: Use 'Pipeline' instead of 'pipeline.Pipeline'
        ('num', Pipeline(num_pipeline), numerical_cols), 
        ('cat', Pipeline(cat_pipeline), categorical_cols)
    ],
    remainder='passthrough',
    verbose_feature_names_out=True
).set_output(transform="pandas")

# 3. Fit and Transform
X_train_processed = full_preprocessor.fit_transform(X_train)

print(f"Pre-processing complete.")
print(f"X_train_processed shape: {X_train_processed.shape}")
print("-" * 70)

STEP 2: FULL PREPROCESSING (Imputation, Scaling, Encoding)
Pre-processing complete.
X_train_processed shape: (204277, 31)
----------------------------------------------------------------------


In [11]:
# ----------------------------------------------------------------------
# --- STEP 3: FEATURE SELECTION (RANDOM FOREST IMPORTANCE) ---
# ----------------------------------------------------------------------

print("=" * 70)
print("STEP 3: EMBEDDED FEATURE SELECTION (Random Forest Importance)")
print("=" * 70)

# 1. Define the base model
tree_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)

# 2. Use SelectFromModel to automatically select features
sfm_tree = SelectFromModel(tree_model, threshold='mean', prefit=False)
sfm_tree.fit(X_train_processed, y_train_encoded)

# 3. Get results
selected_features_tree = X_train_processed.columns[sfm_tree.get_support()]

# FIX: Access the fitted model via sfm_tree.estimator_
fitted_tree_model = sfm_tree.estimator_
feature_importances = pd.Series(
    fitted_tree_model.feature_importances_, 
    index=X_train_processed.columns
)
mean_importance = feature_importances.mean()

# 4. Detailed Output
print(f"Total features before selection: {X_train_processed.shape[1]}")
print(f"Selection Threshold (Mean Importance): {mean_importance:.6f}")
print("-" * 35)

print(f"✅ Features retained: {len(selected_features_tree)}")
print(f"Features removed: {X_train_processed.shape[1] - len(selected_features_tree)}")

print("\n[TOP 10 FEATURES BY IMPORTANCE]")
print(feature_importances.sort_values(ascending=False).head(10).to_string())

# 5. Final Data Set
X_train_selected = X_train_processed[selected_features_tree]

 

print("\n[FINAL SELECTED FEATURES]")
print(f"New Training Data Shape: {X_train_selected.shape}")
# ----------------------------------------------------------------------
# --- STEP 3: FEATURE SELECTION (RANDOM FOREST IMPORTANCE) ---
# ----------------------------------------------------------------------

print("=" * 70)
print("STEP 3: EMBEDDED FEATURE SELECTION (Random Forest Importance)")
print("=" * 70)

# 1. Define the base model
tree_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)

# 2. Use SelectFromModel to automatically select features
sfm_tree = SelectFromModel(tree_model, threshold='mean', prefit=False)
sfm_tree.fit(X_train_processed, y_train_encoded)

# 3. Get results
selected_features_tree = X_train_processed.columns[sfm_tree.get_support()]

# FIX: Access the fitted model via sfm_tree.estimator_
fitted_tree_model = sfm_tree.estimator_
feature_importances = pd.Series(
    fitted_tree_model.feature_importances_, 
    index=X_train_processed.columns
)
mean_importance = feature_importances.mean()

# 4. Detailed Output
print(f"Total features before selection: {X_train_processed.shape[1]}")
print(f"Selection Threshold (Mean Importance): {mean_importance:.6f}")
print("-" * 35)

print(f"✅ Features retained: {len(selected_features_tree)}")
print(f"Features removed: {X_train_processed.shape[1] - len(selected_features_tree)}")

print("\n[TOP 10 FEATURES BY IMPORTANCE]")
print(feature_importances.sort_values(ascending=False).head(10).to_string())

# 5. Final Data Set
X_train_selected = X_train_processed[selected_features_tree]
# This line now works because X_test_processed is assumed to be defined by Step 2


print("\n[FINAL SELECTED FEATURES]")
print(f"New Training Data Shape: {X_train_selected.shape}")
print(f"New Test Data Shape: {X_test_selected.shape}")

print("-" * 70)

print("-" * 70)

STEP 3: EMBEDDED FEATURE SELECTION (Random Forest Importance)
Total features before selection: 31
Selection Threshold (Mean Importance): 0.032258
-----------------------------------
✅ Features retained: 8
Features removed: 23

[TOP 10 FEATURES BY IMPORTANCE]
num__Income                    0.119539
num__InterestRate              0.113678
num__LoanAmount                0.107206
num__Age                       0.098514
num__CreditScore               0.094491
num__MonthsEmployed            0.093793
num__DTIRatio                  0.083809
num__LoanTerm                  0.038430
num__NumCreditLines            0.031383
cat__MaritalStatus_Divorced    0.012028

[FINAL SELECTED FEATURES]
New Training Data Shape: (204277, 8)
STEP 3: EMBEDDED FEATURE SELECTION (Random Forest Importance)
Total features before selection: 31
Selection Threshold (Mean Importance): 0.032258
-----------------------------------
✅ Features retained: 8
Features removed: 23

[TOP 10 FEATURES BY IMPORTANCE]
num__Income       

NameError: name 'X_test_selected' is not defined